# Simple Phoneme Classification with Frozen ECoG Encoder

1. Load pretrained ECoG decoder
2. Freeze it and extract x_common
3. Train a simple classifier head

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm

from networks import ECOG_DECODER
from simple_classifier import SimplePhonemeClassifier

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 2. Load Pretrained ECoG Decoder

In [ ]:
# Initialize ecog decoder (same config as training)
ecog_decoder = ECOG_DECODER["ECoGMapping_RNN"](
    n_mels=128,
    n_formants=6,
    n_formants_noise=1,
    network_db=False,
    causal=False,
    anticausal=False,
    n_classes=32
)

# Load pretrained weights
checkpoint_path = "output/e2a/model_epoch90_HB02.pth"  # UPDATE THIS
checkpoint = torch.load(checkpoint_path, map_location='cpu')

# This contains ALL weights before x_common: base_model.* + motion_projection.*
ecog_decoder_weights = checkpoint['models']['ecog_decoder']  # <-- Weights for ecog -> x_common

# Load the weights
state_dict = {k.replace('module.', ''): v for k, v in ecog_decoder_weights.items()}
ecog_decoder.load_state_dict(state_dict, strict=False)
ecog_decoder.to(device)

# FREEZE everything before x_common (base_model + motion_projection)
for param in ecog_decoder.parameters():
    param.requires_grad = False

ecog_decoder.eval()
print("✓ Loaded and froze pretrained ECoG decoder")

## 3. Create Classifier Head

In [ ]:
NUM_PHONEMES = 9  # Change to your number of classes

classifier = SimplePhonemeClassifier(
    input_channels=32,  # x_common has 32 channels
    num_classes=NUM_PHONEMES,
    hidden_dim=128,
    dropout=0.3
).to(device)

print(f"✓ Created classifier: {sum(p.numel() for p in classifier.parameters())} params")

## 4. Prepare Your Data

In [ ]:
from sklearn.model_selection import train_test_split

class PhonemeDataset(Dataset):
    def __init__(self, ecog_list, labels):
        """
        Args:
            ecog_list: list/array of ECoG data, each [electrode_x, electrode_y, time]
            labels: list/array of integers, one per sample
        """
        self.ecog_list = ecog_list
        self.labels = labels
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        ecog = self.ecog_list[idx]  # [electrode_x, electrode_y, time]
        
        # Reshape from [electrode_x, electrode_y, time] to [time, electrode_x*electrode_y]
        elec_x, elec_y, time_steps = ecog.shape
        ecog_flat = ecog.reshape(elec_x * elec_y, time_steps).T  # [time, channels]
        
        return {
            'ecog': torch.FloatTensor(ecog_flat),
            'label': torch.LongTensor([self.labels[idx]])
        }

# ============ LOAD YOUR DATA HERE ============
ecog_raw = np.load('data/S14_ECoG.npy')      # Shape: [n_trials, electrode_x, electrode_y, time]
labels = np.load('data/S14_labels.npy')       # Shape: [n_trials]

# ============ SHIFT LABELS TO START FROM 0 ============
labels = labels - 1
print(f"Label range: {labels.min()} to {labels.max()}")

# ============ CHECK ELECTRODE CONFIG ============
print(f"Original data shape: {ecog_raw.shape}")
print(f"Electrode grid: {ecog_raw.shape[1]} x {ecog_raw.shape[2]} = {ecog_raw.shape[1] * ecog_raw.shape[2]} electrodes")

# ============ SUBSAMPLE TO 8x8 IF NEEDED ============
# Model expects 8x8 = 64 electrodes
TARGET_GRID = (8, 8)
if ecog_raw.shape[1:3] != TARGET_GRID:
    # Subsample columns: take every other column (0, 2, 4, 6, 8, 10, 12, 14)
    ecog_raw = ecog_raw[:, :, ::2, :]  # Take 1st, 3rd, 5th... columns
    print(f"Subsampled to: {ecog_raw.shape}")
    print(f"New electrode grid: {ecog_raw.shape[1]} x {ecog_raw.shape[2]} = {ecog_raw.shape[1] * ecog_raw.shape[2]} electrodes")

# ============ TRAIN/VAL/TEST SPLIT ============
# First split: train+val (80%) vs test (20%)
train_val_ecog, test_ecog, train_val_labels, test_labels = train_test_split(
    ecog_raw, labels, test_size=0.2, random_state=42, stratify=labels
)

# Second split: train (1-test_size of 80%) vs val (test_size of 80%)
train_ecog, val_ecog, train_labels, val_labels = train_test_split(
    train_val_ecog, train_val_labels, test_size=0.5, random_state=42, stratify=train_val_labels
)

# Create datasets and loaders
train_dataset = PhonemeDataset(list(train_ecog), train_labels)
val_dataset = PhonemeDataset(list(val_ecog), val_labels)
test_dataset = PhonemeDataset(list(test_ecog), test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"\nTrain samples: {len(train_dataset)} ({len(train_dataset)/len(ecog_raw):.0%})")
print(f"Val samples: {len(val_dataset)} ({len(val_dataset)/len(ecog_raw):.0%})")
print(f"Test samples: {len(test_dataset)} ({len(test_dataset)/len(ecog_raw):.0%})")

## 5. Having Train + Val + Test, hyperparams chosen based on Val, tested on Test

In [ ]:
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from itertools import product
import matplotlib.pyplot as plt

# ============ EXTRACT x_common FEATURES ============
def extract_features(ecog_decoder, loader, device):
    """Extract x_common features from frozen encoder"""
    ecog_decoder.eval()
    all_features = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting features"):
            ecog = batch['ecog'].to(device)
            labels = batch['label'].squeeze().numpy()
            
            x_common = ecog_decoder(ecog, return_latent=True)  # [B, 32, T]
            
            # Pool over time: mean → [B, 32]
            features = x_common.mean(dim=2).cpu().numpy()
            
            all_features.append(features)
            all_labels.extend(labels)
    
    return np.vstack(all_features), np.array(all_labels)

def extract_raw_features(loader):
    """Extract raw ECoG features (no encoder)"""
    all_features = []
    all_labels = []
    
    for batch in tqdm(loader, desc="Extracting raw features"):
        ecog = batch['ecog'].numpy()  # [B, T, C]
        labels = batch['label'].squeeze().numpy()
        
        # Pool over time: mean → [B, C]
        features = ecog.mean(axis=1)
        
        all_features.append(features)
        all_labels.extend(labels)
    
    return np.vstack(all_features), np.array(all_labels)

# ============ EXTRACT FEATURES ============
print("=" * 50)
print("EXTRACTING FEATURES")
print("=" * 50)

# Encoder features
print("\n[1] Extracting encoder (x_common) features...")
X_train_enc, y_train = extract_features(ecog_decoder, train_loader, device)
X_val_enc, y_val = extract_features(ecog_decoder, val_loader, device)
X_test_enc, y_test = extract_features(ecog_decoder, test_loader, device)
print(f"Encoder features shape: train={X_train_enc.shape}, val={X_val_enc.shape}, test={X_test_enc.shape}")

# Raw features (no encoder)
print("\n[2] Extracting raw ECoG features (no encoder)...")
X_train_raw, _ = extract_raw_features(train_loader)
X_val_raw, _ = extract_raw_features(val_loader)
X_test_raw, _ = extract_raw_features(test_loader)
print(f"Raw features shape: train={X_train_raw.shape}, val={X_val_raw.shape}, test={X_test_raw.shape}")

# Class distribution
print("\nClass distribution (train):")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u}: {c} samples ({c/len(y_train):.1%})")

# ============ STANDARDIZE ============
# Fit scaler on train only, transform all
scaler_raw = StandardScaler()
X_train_raw_scaled = scaler_raw.fit_transform(X_train_raw)
X_val_raw_scaled = scaler_raw.transform(X_val_raw)
X_test_raw_scaled = scaler_raw.transform(X_test_raw)

scaler_enc = StandardScaler()
X_train_enc_scaled = scaler_enc.fit_transform(X_train_enc)
X_val_enc_scaled = scaler_enc.transform(X_val_enc)
X_test_enc_scaled = scaler_enc.transform(X_test_enc)

# Combine train+val for final training (after grid search)
X_trainval_raw_scaled = np.vstack([X_train_raw_scaled, X_val_raw_scaled])
X_trainval_enc_scaled = np.vstack([X_train_enc_scaled, X_val_enc_scaled])
y_trainval = np.concatenate([y_train, y_val])

print(f"\nTrain+Val combined: {len(y_trainval)} samples")

# ============ GRID SEARCH USING VAL SET ============
def grid_search_val(X_train, y_train, X_val, y_val, X_trainval, y_trainval, pipe, param_grid, name):
    """
    Grid search using validation set:
    1. Search: train on train, evaluate on val to find best params
    2. Final: retrain on train+val with best params
    """
    best_val_acc = 0
    best_params = None
    
    # Generate all param combinations
    keys = list(param_grid.keys())
    values = list(param_grid.values())
    
    print(f"\nSearching {len(list(product(*values)))} combinations...")
    
    for combo in product(*values):
        params = dict(zip(keys, combo))
        
        # Clone pipeline and set params
        model = Pipeline(pipe.steps)
        model.set_params(**params)
        
        # Fit on train, evaluate on val
        model.fit(X_train, y_train)
        val_acc = accuracy_score(y_val, model.predict(X_val))
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_params = params
    
    print(f"Best val acc: {best_val_acc:.2%}")
    print(f"Best params: {best_params}")
    
    # Retrain on train+val with best params
    final_model = Pipeline(pipe.steps)
    final_model.set_params(**best_params)
    final_model.fit(X_trainval, y_trainval)
    print(f"Retrained on train+val ({len(y_trainval)} samples)")
    
    return final_model, best_params, best_val_acc

# ============ PARAM GRIDS ============
param_grid_pca_svm = {
    'pca__n_components': [0.7, 0.8, 0.9, 0.95],
    'pca__whiten': [False, True],
    'svm__C': [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1],
    'svm__kernel': ['linear'],
    'svm__class_weight': ['balanced']
}

param_grid_pca_lda = {
    'pca__n_components': [0.7, 0.8, 0.9, 0.95],
    'pca__whiten': [False, True],
    'lda__solver': ['svd'],
}

results = {}

# ============ METHOD 1: RAW + PCA + SVM ============
print("\n" + "=" * 50)
print("METHOD 1: RAW + PCA + SVM")
print("=" * 50)

pipe = Pipeline([('pca', PCA()), ('svm', SVC(random_state=42))])
model, params, val_acc = grid_search_val(
    X_train_raw_scaled, y_train, X_val_raw_scaled, y_val,
    X_trainval_raw_scaled, y_trainval, pipe, param_grid_pca_svm, "Raw+PCA+SVM"
)
test_acc = accuracy_score(y_test, model.predict(X_test_raw_scaled))
results['Raw+PCA+SVM'] = {'val': val_acc, 'test': test_acc, 'model': model, 'data': 'raw', 'params': params}
print(f"TEST Acc: {test_acc:.2%}")

# ============ METHOD 2: RAW + PCA + LDA ============
print("\n" + "=" * 50)
print("METHOD 2: RAW + PCA + LDA")
print("=" * 50)

pipe = Pipeline([('pca', PCA()), ('lda', LinearDiscriminantAnalysis())])
model, params, val_acc = grid_search_val(
    X_train_raw_scaled, y_train, X_val_raw_scaled, y_val,
    X_trainval_raw_scaled, y_trainval, pipe, param_grid_pca_lda, "Raw+PCA+LDA"
)
test_acc = accuracy_score(y_test, model.predict(X_test_raw_scaled))
results['Raw+PCA+LDA'] = {'val': val_acc, 'test': test_acc, 'model': model, 'data': 'raw', 'params': params}
print(f"TEST Acc: {test_acc:.2%}")

# ============ METHOD 3: ENC + PCA + SVM ============
print("\n" + "=" * 50)
print("METHOD 3: ENCODER + PCA + SVM")
print("=" * 50)

pipe = Pipeline([('pca', PCA()), ('svm', SVC(random_state=42))])
model, params, val_acc = grid_search_val(
    X_train_enc_scaled, y_train, X_val_enc_scaled, y_val,
    X_trainval_enc_scaled, y_trainval, pipe, param_grid_pca_svm, "Enc+PCA+SVM"
)
test_acc = accuracy_score(y_test, model.predict(X_test_enc_scaled))
results['Enc+PCA+SVM'] = {'val': val_acc, 'test': test_acc, 'model': model, 'data': 'enc', 'params': params}
print(f"TEST Acc: {test_acc:.2%}")

# ============ METHOD 4: ENC + PCA + LDA ============
print("\n" + "=" * 50)
print("METHOD 4: ENCODER + PCA + LDA")
print("=" * 50)

pipe = Pipeline([('pca', PCA()), ('lda', LinearDiscriminantAnalysis())])
model, params, val_acc = grid_search_val(
    X_train_enc_scaled, y_train, X_val_enc_scaled, y_val,
    X_trainval_enc_scaled, y_trainval, pipe, param_grid_pca_lda, "Enc+PCA+LDA"
)
test_acc = accuracy_score(y_test, model.predict(X_test_enc_scaled))
results['Enc+PCA+LDA'] = {'val': val_acc, 'test': test_acc, 'model': model, 'data': 'enc', 'params': params}
print(f"TEST Acc: {test_acc:.2%}")

# ============ COMPARISON ============
print("\n" + "=" * 50)
print("COMPARISON")
print("=" * 50)
print("Grid search: train on Train, tune on Val")
print("Final model: retrain on Train+Val, evaluate on Test")
print("-" * 50)
print(f"{'Method':<20} {'Val Acc':<12} {'Test Acc':<12}")
print("-" * 44)
for method, scores in results.items():
    print(f"{method:<20} {scores['val']:.2%}        {scores['test']:.2%}")

# Best method by test accuracy
best_method = max(results, key=lambda x: results[x]['test'])
print(f"\nBest method: {best_method} (Test Acc: {results[best_method]['test']:.2%})")

# ============ CONFUSION MATRIX FOR BEST ============
best_model = results[best_method]['model']
if results[best_method]['data'] == 'raw':
    y_test_pred = best_model.predict(X_test_raw_scaled)
else:
    y_test_pred = best_model.predict(X_test_enc_scaled)

cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'{best_method} - Test Confusion Matrix (Acc: {results[best_method]["test"]:.2%})')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center')
plt.show()

print("\nClassification Report (Test):")
print(classification_report(y_test, y_test_pred, zero_division=0))

## 6. Training Loop

In [ ]:
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

# ============ EXTRACT x_common FEATURES ============
def extract_features(ecog_decoder, loader, device):
    """Extract x_common features from frozen encoder"""
    ecog_decoder.eval()
    all_features = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting features"):
            ecog = batch['ecog'].to(device)
            labels = batch['label'].squeeze().numpy()
            
            x_common = ecog_decoder(ecog, return_latent=True)  # [B, 32, T]
            
            # Pool over time: mean → [B, 32]
            features = x_common.mean(dim=2).cpu().numpy()
            
            all_features.append(features)
            all_labels.extend(labels)
    
    return np.vstack(all_features), np.array(all_labels)

def extract_raw_features(loader):
    """Extract raw ECoG features (no encoder)"""
    all_features = []
    all_labels = []
    
    for batch in tqdm(loader, desc="Extracting raw features"):
        ecog = batch['ecog'].numpy()  # [B, T, C]
        labels = batch['label'].squeeze().numpy()
        
        # Pool over time: mean → [B, C]
        features = ecog.mean(axis=1)
        
        all_features.append(features)
        all_labels.extend(labels)
    
    return np.vstack(all_features), np.array(all_labels)

In [ ]:
# ============ EXTRACT FEATURES ============
print("=" * 50)
print("EXTRACTING FEATURES")
print("=" * 50)

# Encoder features
print("\n[1] Extracting encoder (x_common) features...")
X_train_enc, y_train = extract_features(ecog_decoder, train_loader, device)
X_val_enc, y_val = extract_features(ecog_decoder, val_loader, device)
print(f"Encoder features shape: {X_train_enc.shape}")

# Raw features (no encoder)
print("\n[2] Extracting raw ECoG features (no encoder)...")
X_train_raw, _ = extract_raw_features(train_loader)
X_val_raw, _ = extract_raw_features(val_loader)
print(f"Raw features shape: {X_train_raw.shape}")

# Class distribution
print("\nClass distribution (train):")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u}: {c} samples ({c/len(y_train):.1%})")

n_classes = len(np.unique(y_train))

In [ ]:
# ============ PCA + SVM GRID ============
param_grid_pca_svm = {
    'pca__n_components': [0.7, 0.8, 0.9, 0.95],
    'pca__whiten': [False, True],
    'svm__C': [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1],
    'svm__kernel': ['linear'],
    'svm__class_weight': ['balanced']
}

# ============ PCA + LDA GRID ============
param_grid_pca_lda = {
    'pca__n_components': [0.7, 0.8, 0.9, 0.95],
    'pca__whiten': [False, True],
    'lda__solver': ['svd'],
}

results = {}

In [ ]:
# ============ METHOD 1: RAW ECoG + PCA + SVM ============
print("\n" + "=" * 50)
print("METHOD 1: RAW ECoG + PCA + SVM")
print("=" * 50)

scaler_raw = StandardScaler()
X_train_raw_scaled = scaler_raw.fit_transform(X_train_raw)
X_val_raw_scaled = scaler_raw.transform(X_val_raw)

pipe_raw_pca_svm = Pipeline([
    ('pca', PCA()),
    ('svm', SVC(random_state=42))
])

grid_raw_pca_svm = GridSearchCV(pipe_raw_pca_svm, param_grid_pca_svm, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_raw_pca_svm.fit(X_train_raw_scaled, y_train)

train_acc = accuracy_score(y_train, grid_raw_pca_svm.predict(X_train_raw_scaled))
val_acc = accuracy_score(y_val, grid_raw_pca_svm.predict(X_val_raw_scaled))
results['Raw + PCA + SVM'] = {'train': train_acc, 'val': val_acc, 'cv': grid_raw_pca_svm.best_score_, 'model': grid_raw_pca_svm, 'data': 'raw'}

print(f"Best params: {grid_raw_pca_svm.best_params_}")
print(f"CV score: {grid_raw_pca_svm.best_score_:.2%}")
print(f"Train Acc: {train_acc:.2%} | Val Acc: {val_acc:.2%}")

In [ ]:
# ============ METHOD 2: RAW ECoG + PCA + LDA ============
print("\n" + "=" * 50)
print("METHOD 2: RAW ECoG + PCA + LDA")
print("=" * 50)

pipe_raw_pca_lda = Pipeline([
    ('pca', PCA()),
    ('lda', LinearDiscriminantAnalysis())
])

grid_raw_pca_lda = GridSearchCV(pipe_raw_pca_lda, param_grid_pca_lda, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_raw_pca_lda.fit(X_train_raw_scaled, y_train)

train_acc = accuracy_score(y_train, grid_raw_pca_lda.predict(X_train_raw_scaled))
val_acc = accuracy_score(y_val, grid_raw_pca_lda.predict(X_val_raw_scaled))
results['Raw + PCA + LDA'] = {'train': train_acc, 'val': val_acc, 'cv': grid_raw_pca_lda.best_score_, 'model': grid_raw_pca_lda, 'data': 'raw'}

print(f"Best params: {grid_raw_pca_lda.best_params_}")
print(f"CV score: {grid_raw_pca_lda.best_score_:.2%}")
print(f"Train Acc: {train_acc:.2%} | Val Acc: {val_acc:.2%}")

In [ ]:
# ============ METHOD 3: ENCODER + PCA + SVM ============
print("\n" + "=" * 50)
print("METHOD 3: ENCODER + PCA + SVM")
print("=" * 50)

scaler_enc = StandardScaler()
X_train_enc_scaled = scaler_enc.fit_transform(X_train_enc)
X_val_enc_scaled = scaler_enc.transform(X_val_enc)

pipe_enc_pca_svm = Pipeline([
    ('pca', PCA()),
    ('svm', SVC(random_state=42))
])

grid_enc_pca_svm = GridSearchCV(pipe_enc_pca_svm, param_grid_pca_svm, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_enc_pca_svm.fit(X_train_enc_scaled, y_train)

train_acc = accuracy_score(y_train, grid_enc_pca_svm.predict(X_train_enc_scaled))
val_acc = accuracy_score(y_val, grid_enc_pca_svm.predict(X_val_enc_scaled))
results['Enc + PCA + SVM'] = {'train': train_acc, 'val': val_acc, 'cv': grid_enc_pca_svm.best_score_, 'model': grid_enc_pca_svm, 'data': 'enc'}

print(f"Best params: {grid_enc_pca_svm.best_params_}")
print(f"CV score: {grid_enc_pca_svm.best_score_:.2%}")
print(f"Train Acc: {train_acc:.2%} | Val Acc: {val_acc:.2%}")

In [ ]:
# ============ METHOD 4: ENCODER + PCA + LDA ============
print("\n" + "=" * 50)
print("METHOD 4: ENCODER + PCA + LDA")
print("=" * 50)

pipe_enc_pca_lda = Pipeline([
    ('pca', PCA()),
    ('lda', LinearDiscriminantAnalysis())
])

grid_enc_pca_lda = GridSearchCV(pipe_enc_pca_lda, param_grid_pca_lda, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_enc_pca_lda.fit(X_train_enc_scaled, y_train)

train_acc = accuracy_score(y_train, grid_enc_pca_lda.predict(X_train_enc_scaled))
val_acc = accuracy_score(y_val, grid_enc_pca_lda.predict(X_val_enc_scaled))
results['Enc + PCA + LDA'] = {'train': train_acc, 'val': val_acc, 'cv': grid_enc_pca_lda.best_score_, 'model': grid_enc_pca_lda, 'data': 'enc'}

print(f"Best params: {grid_enc_pca_lda.best_params_}")
print(f"CV score: {grid_enc_pca_lda.best_score_:.2%}")
print(f"Train Acc: {train_acc:.2%} | Val Acc: {val_acc:.2%}")

In [ ]:
# ============ COMPARISON ============
print("\n" + "=" * 50)
print("COMPARISON")
print("=" * 50)
print(f"{'Method':<20} {'CV Score':<12} {'Train Acc':<12} {'Val Acc':<12}")
print("-" * 56)
for method, scores in results.items():
    print(f"{method:<20} {scores['cv']:.2%}        {scores['train']:.2%}        {scores['val']:.2%}")

# Best method by validation accuracy
best_method = max(results, key=lambda x: results[x]['val'])
print(f"\nBest method: {best_method} (Val Acc: {results[best_method]['val']:.2%})")

In [ ]:
# ============ CONFUSION MATRIX FOR BEST ============
best_model = results[best_method]['model']
if results[best_method]['data'] == 'raw':
    y_val_pred = best_model.predict(X_val_raw_scaled)
else:
    y_val_pred = best_model.predict(X_val_enc_scaled)

cm = confusion_matrix(y_val, y_val_pred)
plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'{best_method} - Confusion Matrix (Val Acc: {results[best_method]["val"]:.2%})')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center')
plt.show()

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred, zero_division=0))